In [ ]:
"""
Purpose:
    - KRX 거래일 마스터 생성 (pandas_market_calendars 기반)

✨ v3.9.2 패치 (2026-03-16):
- if __name__ == "__main__" 가드 제거 → 함수 정의 셀 / 실행 셀 분리
- get_krx_trading_days(): warnings.catch_warnings()로 discontinued 경고 억제
"""

import os
import pandas as pd
from pathlib import Path
from datetime import datetime
from src.utils.config import load_config

def get_krx_trading_days(start_date: str, end_date: str) -> pd.DataFrame:
    """
    전문 라이브러리를 사용하여 KRX 거래일 획득 시도 (Fallback 로직 포함)
    """
    # 1. pandas_market_calendars 시도
    try:
        import warnings
        import pandas_market_calendars as mcal
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                message='.*discontinued.*',
                category=UserWarning
            )
            krx = mcal.get_calendar('XKRX')
        schedule = krx.schedule(start_date=start_date, end_date=end_date)
        dates = pd.to_datetime(schedule.index).date
        return pd.DataFrame({'date': dates})
    except Exception as e:
        print(f"⚠️ pandas_market_calendars 사용 실패, 대체 로직으로 전환: {e}")
        
    # 2. Fallback: holidays 라이브러리 기반 추정
    try:
        import holidays
        start = pd.to_datetime(start_date).date()
        end = pd.to_datetime(end_date).date()
        all_days = pd.date_range(start=start, end=end, freq='D')
        weekdays = all_days[all_days.dayofweek < 5]
        kr_holidays = holidays.KR(years=range(start.year, end.year + 1))
        trading_days = [d.date() for d in weekdays if d.date() not in kr_holidays]
        return pd.DataFrame({'date': trading_days})
    except ImportError:
        raise RuntimeError("대체 로직을 위해 'holidays' 패키지가 필요합니다. pip install holidays")

def update_market_calendar():
    """
    설정 파일의 기간에 맞춰 캘린더 생성 및 저장
    """
    cfg = load_config()
    
    # 1. 설정 및 경로 확보
    # config.yaml에 calendar 섹션이 없으면 기본값 사용
    cal_cfg = cfg.get('calendar', {})
    start_date = cal_cfg.get('start_date', cfg['data_collection']['start_date'])
    end_date = cal_cfg.get('end_date', cfg['data_collection']['end_date'])
    
    meta_dir = Path(cfg['paths'].get('meta_dir', 'data/99_meta'))
    meta_dir.mkdir(parents=True, exist_ok=True)
    save_path = meta_dir / "krx_calendar.csv"

    print(f"📅 캘린더 업데이트 중: {start_date} ~ {end_date}")
    
    # 2. 날짜 계산 및 저장
    df = get_krx_trading_days(start_date, end_date)
    df = df.sort_values('date').reset_index(drop=True)
    
    df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"✅ 캘린더 저장 완료: {save_path.absolute()} (총 {len(df)}일)")
    
    return df


In [ ]:
update_market_calendar()